# Part 1: Data Cleaning for Market Trend Discovery System with LLMs and Clustering

**Objective:**
- Prepare and clean airline-related Twitter comments to generate high-quality embeddings for downstream clustering and trend discovery.
- Clean and preproccessed complementary dataset features to prepare data for trend analysis. 

**Description:**
- In this stage, the raw Twitter comments dataset is cleaned and normalize to remove noise, normalize text, and prepare it for embedding generation. This step ensures that the textual data is consistent and suitable for semantic analysis using LLM-based embeddinigs and clustering techniques. Additionally, the dataset contains complementary features for trend analysis, including both numerical and categorical variables. These features are processed and standardized to ensure consistency and usability for downstream analysos such as clustering and market trend discovery. 

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder

In [ ]:
columns = ['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence', 'negativereason',
           'negativereason_confidence', 'airline', 'airline_sentiment_gold', 'name',
           'negativereason_gold', 'retweet_count', 'text', 'tweet_coord', 'tweet_created',
           'tweet_location', 'user_timezone']

df = pd.read_csv('dataset.csv', header=None, names=columns, index_col=False)
df.head()

In [ ]:
# Check for missing values 
df.info()
print(df.shape)

There are no missing values to account because every line has at least 1 character. Except for tweet_coord and tweet_created

In [ ]:
# Check for the numerical features
df.describe()

- Approximately 25% of the observations have sentiment confidence below 0.69, indicating higher labeling uncertainty that could introduce noise into downstream models. 
- The retweet_count variable is highly zero-inflated (median = 0), suggesting that most customer complaints do not propagate widely on Twitter. While this may limit its direct predictive power, further analysis will evaluate whether tweets with any retweet activity exhibit different sentiment or thematic patterns.

In [ ]:
# check for missing values 
num_col = df.select_dtypes(include=["int64", "float64"])

num_col.isna().sum()

In [ ]:
# Check for the object features
df.describe(include="object")

- Columns present a lot of missing values: negative_reason, negativereason_confidence, airline_sentiment_gold, negativereason_gold, tweet_coord, tweet_created, tweet_location, user_timezone.
- negativereason_confidence has 1411 unique values, which is almost the same amount as the total samples. Meaninig this could be a numerical variable. 
- name also has too many unique values, this could mean that there might not be a naming convention.

In [ ]:
df_new = df.copy()

In [ ]:
# Review changes for categorical columns. 
def review_uniques(data_type):
    if data_type == "cat":
        cols = df_new.select_dtypes(include="object").columns
    else: 
        cols = df_new.select_dtypes(include=["float64", "int64"]).columns
        
    for col in cols: 
        print(df_new[col].value_counts())
        print()

review_uniques("cat")

Features that need transformations: 
- airline needs to be encoded because it is textual information that is not particularly implied in the text.
- negativereason has 5642 missing values, impute missing values with "missing".
- negativereason_confidence, has numerical values. The variable must be transformed to analyze its distribution and take de decision on how to treat missing values.

Features to be eliminated: 
- airline_sentiment_gold has 14600 missing values out of 14640 (aprox. 99%).
- negativereason_gold, presents the same issue as airline_sentiment_gold. 
- name is the user name which made the comment, this can introduce noise, and it is not an explicative feature. Thus the variable will also be eliminated.

Features with mixed information: tweet_coord, tweet_created, tweet_location, user_timezone. Additionally: 
- tweet_location, presents 4121 missing data (aprox. 28%)
- user_timezone, presents 3990 missing data (approx. 27%)

However let's get a clearer picture about the pct of missingness each category presents. 

In [ ]:
# let's rename the missing values with nan to obtain a clear count of missingness of our mixed data.
cat_cols = df_new.select_dtypes(include="object").columns

for col in cat_cols:
    df_new[col] = (df_new[col]
                   .astype(str)
                   .str.strip()
                   .replace("?", np.nan)
    )
    missing_pct = round((df_new[col].isna().sum()/df_new.shape[0])*100,2)
    if missing_pct > 50: 
        print(col)

In [ ]:
# eliminate columns with higly missing values pct or the ones that are irrelevant to the analysis. 
cols_to_delete = ["tweet_id","airline_sentiment_gold", "negativereason_gold", "name", "tweet_coord"]
df_new = df_new.drop(columns=cols_to_delete)
df_new.shape

In [ ]:
# transform airline variable
dummies = pd.get_dummies(df_new["airline"], prefix="airline")
df_new = pd.concat([df_new, dummies], axis=1)
df_new.shape

In [ ]:
# Transform negativereason_confidence into float. 
df_new["negativereason_confidence"] = df_new["negativereason_confidence"].astype(float)

In [ ]:
# Check distributions
num_cols = df_new.select_dtypes(include=["int64", "float64"]).columns
n_cols = 2
n_rows = math.ceil(len(num_cols)/n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df_new[col], kde=True, ax=axes[i])
    axes[i].set_title(col)

# delete empty axes
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

negativereason_confidence shows a multimodal and right-biased distribution. Before imputing any value, let's understand if these missing values are linked to positive reasons. 

In [ ]:
df_new["negativereason_confidence"].describe()
pd.crosstab(
    df_new["airline_sentiment"],
    df_new["negativereason_confidence"].isna()
)

based on the table, negative sentiment comments have always a value of confidence, while the neutral or postive ones, have a lot of the times NA. Which means this is an informative missing value and not random missing vlaues. 
This requires to insert a missing flag for the values and impute zero as confidence value which will mark that there is no evidence that the comment is either neutral or positive. 

In [ ]:
# add missingness flag
df_new["negativereason_confidence_flag"] = df_new["negativereason_confidence"].isna().astype(int)
df_new["negativereason_confidence"] = df_new["negativereason_confidence"].fillna(0)

In [ ]:
# check for outliers only for numerical features. 
num_cols = df_new.select_dtypes(include=["float64", "int64"])

# eliminate non-numeric columns for the outliers analysis (such as the one with "_flag") 
num_cols = list(num_cols.columns[~num_cols.columns.str.contains("flag")])

def iqr_outliers(feature):
    q1 = feature.quantile(0.25)
    q3 = feature.quantile(0.75)

    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return lower, upper 

outlier_summary = {}
for col in num_cols: 
    f = df_new[col]
    lower, upper = iqr_outliers(f)

    n_outliers = ((f < lower) | (f > upper)).sum()
    outlier_summary[col] = {
        "lower": lower, 
        "upper": upper, 
        "n_outliers": n_outliers, 
        "pct_outliers": round((n_outliers/len(f)*100), 2)
    }

# check which features have possible errors, rare values, long tales or they represent subsamples. 
possible_errors = []
rare_values = []
long_tales = []
subsamples = []

for key in outlier_summary.keys():
    filtered_data = outlier_summary[key]["pct_outliers"]
    if (filtered_data < 0.5) and (filtered_data != 0):
        possible_errors.append(key)
    elif (filtered_data > 0.5) and (filtered_data < 2): 
        rare_values.append(key)
    elif (filtered_data > 2) and (filtered_data < 10): 
        long_tales.append(key)
    else: 
        subsamples.append(key)

print(f"Possible features with errors {possible_errors}")
print(f"rare values {rare_values}")
print(f"long tales {long_tales}")
print(f"subsamples {subsamples}")

retweet_count has long_tales, which based on the nature of the feature is normal, because there are lots of zeros and a little amount of high values. However, it is important to treat this issue since we are applying clustering and can be affected by the distances. 

In [ ]:
# plot outliers
def plot_outliers(feature, v_df):
    n_cols = 2
    n_rows = math.ceil(len(feature)/n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(8, 2*n_rows))
    axes = axes.flatten()
    
    for i, col in enumerate(feature):
        sns.boxplot(x=v_df[col], ax=axes[i])
        axes[i].set_title(col)

    for j in range(i+1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

plot_outliers(long_tales, df_new)

In [ ]:
df_new["retweet_count"] = np.log1p(df_new["retweet_count"])
plot_outliers(["retweet_count"], df_new)

Now let's treat the categorical features

In [ ]:
# check is the missingness is related to a sentiment that could provoke data leakage. 
pd.crosstab(
    df_new["negativereason"], 
    df_new["airline_sentiment"]
)

In [ ]:
# replace missing values with "not_applicable"
df_new["negativereason_flag"] = df_new["negativereason"].isna().astype(int)
df_new["negativereason"] = df_new["negativereason"].fillna("not_applicable")

Since the missingness of the negativereason is directly correlated with the sentiment, this could potentially lead to a data leakage. Thus the variable won't be used in the clustering dataset. 

Now let's continue with our mixed variables. For this type of problem, we will parse the data into another structure to treat it. 

the only category columns left are: 'tweet_created', 'tweet_location', 'user_timezone'

In [ ]:
# start with tweet_created. 
# Eliminate extra strings that could interfere with parsing
df_new["tweet_created"] = (df_new["tweet_created"]
    .astype(str)
    .str.strip()
    .str.replace("'", "", regex=False)
)

# transform data into datetime format
df_new["tweet_created_datetime"] = pd.to_datetime(
    df_new["tweet_created"], 
    format="%Y-%m-%d %H:%M:%S %z",
    errors="coerce"
)

# extract hour, dayofweek, weekend and month for temporal analysis.
df_new["tweet_hour"] = df_new["tweet_created_datetime"].dt.hour
df_new["tweet_dayofweek"] = df_new["tweet_created_datetime"].dt.dayofweek
df_new["tweet_is_weekend"] = df_new["tweet_dayofweek"].isin([5,6]).astype(int)
df_new["tweet_month"] = df_new["tweet_created_datetime"].dt.month

In [ ]:
# check how much data was capable of being treated
df_new["tweet_created_datetime"].notna().mean()

In [ ]:
# treat missing values 
df_new["tweet_datetime_missing"] = df_new["tweet_created_datetime"].isna().astype(int)

Since tweet_hour, tweet_dayofweek, tweet_month are numerical, we can now impute missing values by mode without affecting our data. 

In [ ]:
cols_to_impute = ["tweet_hour", "tweet_dayofweek", "tweet_month", "tweet_is_weekend"]

for col in cols_to_impute: 
    df_new[col] = df_new[col].fillna(df_new[col].mode()[0])

df_new["tweet_is_weekend"] = df_new["tweet_dayofweek"].isin([5,6]).astype(int)

df_new = df_new.drop(columns=["tweet_created_datetime", "tweet_created"])
df_new.shape

In [ ]:
# let's go with 'tweet_location'
# normalize strings 
df_new["tweet_location_clean"] = (
    df_new["tweet_location"]
    .astype(str)
    .str.strip()
    .str.replace("'", "", regex=False)
)

# replace category nan for actual nan values
df_new["tweet_location_clean"] = df_new["tweet_location_clean"].replace("nan", np.nan)

# detect trash which is any data that is not reated with location. In this case, datetimes. 
mask_date = df_new["tweet_location_clean"].str.match(r"\d{4}-\d{2}-\d{2}", na=False)
df_new.loc[mask_date, "tweet_location_clean"] = np.nan

# detect values type coordinate
mask_coord_like = df_new["tweet_location_clean"].str.contains(r"\[|\]", na=False)
df_new.loc[mask_coord_like, "tweet_location_clean"] = np.nan

# normalize names
df_new["tweet_location_clean"] = (
    df_new["tweet_location_clean"]
    .str.lower()
    .str.strip()
)

# check how many values were treated
coverage = df_new["tweet_location_clean"].notna().mean()
print(round(coverage,2))

# Check how many unique values 
df_new["tweet_location_clean"].value_counts()

The feature has 2069 unique features which means high dimensionality, semantic noise, bad geographical signal, high sparsity and possible inconsistencies.

In [ ]:
# apply bucketing to reduce high cardinality.
# 30 is based on heuristic social data. 
# top 30 normally captures most of the useful signal
top_n = 30 

top_locs = df_new["tweet_location_clean"].value_counts().nlargest(30).index

df_new["tweet_location_bucket"] = df_new["tweet_location_clean"].where(
    df_new["tweet_location_clean"].isin(top_locs), 
    "other"
)

df_new["tweet_location_bucket"].isna().sum()

# delete "tweet_location_clean" and "tweet_location"
df_new = df_new.drop(columns=["tweet_location", "tweet_location_clean"])
df_new.shape

lastly, its time to treat user_timezone

In [ ]:
# normalize strings 
df_new["user_timezone_clean"] = (
    df_new["user_timezone"]
    .astype(str)
    .str.strip()
    .str.replace("'", "", regex=False)
)

# replace category nan for actual nan values
df_new["user_timezone_clean"] = df_new["user_timezone_clean"].replace("nan", np.nan)

# detect trash which is any data that is not reated with location. In this case, datetimes. 
mask_date = df_new["user_timezone_clean"].str.match(r"\d{4}-\d{2}-\d{2}", na=False)
df_new.loc[mask_date, "user_timezone_clean"] = np.nan

# detect values type coordinate
mask_coord_like = df_new["user_timezone_clean"].str.contains(r"\[|\]", na=False)
df_new.loc[mask_coord_like, "user_timezone_clean"] = np.nan

# normalize names
df_new["user_timezone_clean"] = (
    df_new["user_timezone_clean"]
    .str.lower()
    .str.strip()
)

# check how many values were treated
coverage = df_new["user_timezone_clean"].notna().mean()
print(round(coverage,2))

# Check how many unique values 
df_new["user_timezone_clean"].value_counts()

In [ ]:
round(df_new["user_timezone_clean"].isna().mean(),2)

In [ ]:
df_new["user_timezone_clean"].value_counts().head(30)

freq = df_new['user_timezone_clean'].value_counts(normalize=True).cumsum()
freq.head(50)

Based on the previous table, top 50 captures around 71% of data however in top 30 it captures around 63% of data. which could mean that after that might be noise. Thus top_n = 30 will be best. 

In [ ]:
# apply bucketing to reduce high cardinality.
# treat missing values as missing, because since they represent 32% of data, they could
# introduce different signals. It is important to treat it as a new category. 
df_new["user_timezone_clean"] = df_new["user_timezone_clean"].fillna("missing")

# top 30 normally captures most of the useful signal
top_locs = df_new["user_timezone_clean"].value_counts().nlargest(30).index

df_new["user_timezone_bucket"] = df_new["user_timezone_clean"].where(
    df_new["user_timezone_clean"].isin(top_locs), 
    "other"
)

print(df_new["user_timezone_bucket"].isna().sum())

# delete "user_timezone_clean" and "user_timezone"
df_new = df_new.drop(columns=["user_timezone", "user_timezone_clean"])
df_new.shape

In [ ]:
# Encode tweet_location_bucket and user_timezone_bucket before embeddings. 
cols_to_encode = ["tweet_location_bucket", "user_timezone_bucket"]
names = ["location", "timezone"]

for col, name in zip(cols_to_encode, names):
    dummies = pd.get_dummies(df_new[col], prefix=name)
    df_new = pd.concat([df_new, dummies], axis=1)

df_new.shape

In [ ]:
# drop the encoded columns
df_new = df_new.drop(columns=cols_to_encode)
df_new.shape

In [ ]:
df_new.info()

In [ ]:
# Save cleaned dataset
df_new.to_parquet("tweets_clean.parquet", index=False)